In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.append("../")

In [ ]:
import numpy as np
import pandas as pd

import src.data_preprocessing.caption_generation as cg
import src.data_preprocessing.create_aux_data as cad
import src.data_preprocessing.data_utils as du
import src.data_preprocessing.gee_utils as gu
from src.data.base_caption_builder import BaseCaptionBuilder, DummyCaptionBuilder
from src.data.base_datamodule import BaseDataModule
from src.data.butterfly_caption_builder import ButterflyCaptionBuilder
from src.data.butterfly_dataset import ButterflyDataset

In [ ]:
# Load data

BD = ButterflyDataset(
    modalities={
        "coords": None,
        "s2": {"size": 256, "format": "tif", "channels": "4c", "preprocessing": "none"},
    },
    data_dir=os.path.join(os.environ["DATA_DIR"]),
    use_aux_data=True,
)
BD.setup()

BCB = ButterflyCaptionBuilder(
    data_dir=os.path.join(os.environ["DATA_DIR"], "s2bms"),
    templates_fname="v4.json",
    seed=42,
)

BM = BaseDataModule(
    dataset=BD,
    batch_size=320,
    split_mode="from_file",
    saved_split_file_name=os.path.join(
        os.environ["DATA_DIR"], "s2bms", "splits/split_indices_s2bms_2024-08-14-1459.pth"
    ),
    caption_builder=BCB,
)

batch = next(iter(BM.test_dataloader()))
batch["eo"]["s2"].shape

In [ ]:
batch["text"]

In [ ]:
# load model
import torch

ckpt_path = "../logs/train/runs/2026-02-10_14-09-12/checkpoints/"
assert os.path.exists(ckpt_path), f"Checkpoint path {ckpt_path} does not exist."
name_model = [x for x in os.listdir(ckpt_path) if x.startswith("epoch_")][0]
ckpt_file = os.path.join(ckpt_path, name_model)
print(f"Loading model from checkpoint: {ckpt_file}")
ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)

In [ ]:
ckpt["state_dict"].keys()

In [ ]:
from src.models.components.geo_encoders.cnn_encoder import CNNEncoder
from src.models.components.geo_encoders.geoclip import GeoClipCoordinateEncoder
from src.models.components.loss_fns.bce_loss import BCELoss
from src.models.components.loss_fns.clip_loss import ClipLoss
from src.models.components.pred_heads.mlp_pred_head import MLPPredictionHead
from src.models.components.text_encoders.clip_text_encoder import ClipTextEncoder
from src.models.predictive_model import PredictiveModel
from src.models.text_alignment_model import TextAlignmentModel

model = TextAlignmentModel(
    eo_encoder=GeoClipCoordinateEncoder(),
    text_encoder=ClipTextEncoder(),
    trainable_modules=[],
    optimizer=torch.optim.Adam,
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau,
    loss_fn=ClipLoss,
    num_classes=62,
)

In [ ]:
model.load_state_dict(ckpt["state_dict"], strict=False)
# model.eval()

In [ ]:
tmp = model.forward(batch, mode="val")
tmp[0].shape, tmp[1].shape

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(np.dot(tmp[0].numpy(), tmp[1].numpy().T), vmin=-1, vmax=1, cmap="coolwarm")

In [ ]:
import torch.nn.functional as F


def embed_text(model, txt):
    with torch.no_grad():
        text_tokens = model.processor(text=txt, return_tensors="pt", padding=True)
        device = next(model.model.parameters()).device
        text_tokens = {k: v.to(device) for k, v in text_tokens.items()}

        text_embeds = model.model.get_text_features(**text_tokens)

        # Project
        if model.projector is not None:
            text_embeds = model.projector(text_embeds)

        if model.extra_projector is not None:
            text_embeds = model.extra_projector(text_embeds)

        if model.output_normalization == "l2":
            text_embeds = F.normalize(text_embeds, p=2, dim=-1)
    return text_embeds.detach().cpu().numpy()


eo_feats = tmp[0].detach().cpu().numpy()

In [ ]:
# txt = 'This is a coastal area with a beach, ocean, and the sea.'
txt = "This is a very wooded area with a forest and many trees."
# txt = 'This is an urban area with many buildings and roads, and few trees.'
# txt = 'This is a very fragmented landscape with a mix of urban areas, forests, and agricultural fields.'
feat = embed_text(model.text_encoder, txt)

cosine_sim = np.dot(eo_feats, feat.T).squeeze()
# plt.bar(np.arange(len(cosine_sim)), cosine_sim)

fig, ax = plt.subplots(6, 6, figsize=(20, 20))
n_ims = len(cosine_sim)
n_plots = min(n_ims, 36)
skip_factor = max(1, n_ims // n_plots)
# inds_plot = np.argsort(cosine_sim)[::skip_factor][:n_plots]
# inds_plot = np.argsort(cosine_sim)[-n_plots:]
#
# show a mix of high and low similarity examples
inds_plot = (
    np.argsort(cosine_sim)[: n_plots // 2].tolist()
    + np.argsort(cosine_sim)[-n_plots // 2 :].tolist()
)
ax = ax.flatten()
for i, ind in enumerate(inds_plot):
    ax[i].imshow(batch["eo"]["s2"][ind].permute(1, 2, 0).numpy())
    ax[i].set_title(f"Cosine sim: {cosine_sim[ind]:.2f}")
    ax[i].axis("off")

fig.suptitle(f"Query: '{txt}'", fontsize=16)

In [ ]:
txt_list = [
    "This is a coastal area with a beach, ocean, and the sea.",
    "This is a very wooded area with a forest and many trees.",
    "This is an urban area with many buildings and roads, and few trees.",
    "This is a very biodiverse landscape with birds, butterflies, and many different types of vegetation.",
    # 'This is a very fragmented landscape with a mix of urban, forest, field, crops, bare, beach, and roads.'
]

fig, ax = plt.subplots(2, len(txt_list) // 2, figsize=(10, 10))
ax = ax.flatten()
for i, txt in enumerate(txt_list):
    feat = embed_text(model.text_encoder, txt)

    cosine_sim = np.dot(eo_feats, feat.T).squeeze()
    ind = np.argsort(cosine_sim)[-1]

    ax[i].imshow(batch["eo"]["s2"][ind].permute(1, 2, 0).numpy())
    txt_print = txt[:40] + "\n" + txt[40:] if len(txt) > 40 else txt
    ax[i].set_title(f"Cosine sim: {cosine_sim[ind]:.2f}\n{txt_print}")
    ax[i].axis("off")

In [ ]:
batch["target"][ind]